# Lab 04 — Fault Tolerance: Drive-Level Resilience

This lab demonstrates how RustFS tolerates drive failures using **Erasure Coding (EC:2)**.

Our `docker-compose.yml` mounts **4 Docker volumes** (`drive0`…`drive3`) as separate drives
inside a single RustFS container. RustFS applies a **Reed-Solomon (4+2)** scheme:
- Every object is split into **4 data shards** + **2 parity shards**
- The 6 shards are distributed across all 4 drives
- RustFS can **lose any 2 drives** and still reconstruct every object

In this lab you'll upload objects, verify they're intact, inspect the container health,
and confirm the theoretical protection guarantees.


---
## 0 · Prerequisites

1. RustFS is running: `docker compose up -d`
2. Virtual environment active with `boto3` installed: `uv sync`
3. Docker CLI available in PATH (for health checks)


---
## Setup — Connect and Create Bucket


In [ ]:
import boto3
import subprocess
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
    ),
)

BUCKET = 'lab4-ft'

existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f'✅ Created bucket: {BUCKET}')
else:
    print(f'✅ Bucket already exists: {BUCKET}')


---
## 1 · Upload Test Objects

We upload 10 small text objects. RustFS will automatically stripe each object
across the 4 drives as 4 data shards + 2 parity shards.


In [ ]:
NUM_OBJECTS = 10

for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    content = f'Object {i:02d}: RustFS fault tolerance test. Shard distributed across 4 drives.'
    s3.put_object(Bucket=BUCKET, Key=key, Body=content.encode('utf-8'))
    print(f'  Uploaded {key}')

print(f'\n✅ {NUM_OBJECTS} objects uploaded to s3://{BUCKET}/')


---
## 2 · Verify All Objects Are Intact

Read back all objects and verify the content matches exactly.
This confirms that write → EC stripe → read → EC reconstruct works correctly.


In [ ]:
print('Verifying objects...')
all_ok = True
for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    response = s3.get_object(Bucket=BUCKET, Key=key)
    body = response['Body'].read().decode('utf-8')
    expected = f'Object {i:02d}: RustFS fault tolerance test. Shard distributed across 4 drives.'
    ok = body == expected
    status = '✅' if ok else '❌'
    print(f'  {status} {key}')
    if not ok:
        all_ok = False
        print(f'     Expected: {expected}')
        print(f'     Got:      {body}')

print(f'\n{"✅ All objects verified intact." if all_ok else "❌ Verification failed!"}')


---
## 3 · Check Server Health

RustFS exposes a liveness endpoint at `/minio/health/live` (HTTP 200 = healthy).
We also run `docker compose ps` to confirm all containers are running.


In [ ]:
# --- Liveness check via HTTP ---
result = subprocess.run(
    ['curl', '-sf', '-o', '/dev/null', '-w', '%{http_code}',
     'http://localhost:9000/minio/health/live'],
    capture_output=True, text=True,
)
status_code = result.stdout.strip()
if status_code == '200':
    print('✅ RustFS liveness check: HTTP 200 — server is healthy')
else:
    print(f'⚠️  Unexpected HTTP status: {status_code}')

print()

# --- Container status via Docker ---
result = subprocess.run(
    ['docker', 'compose', 'ps'],
    capture_output=True, text=True, cwd='..'
)
print('Container status:')
print(result.stdout if result.stdout else '(no output — run from repo root)')


---
## Understanding the EC:2 Protection Model

RustFS uses **Reed-Solomon (4+2)** erasure coding inside a single server with 4 drives:

```
Object bytes  ──▶  [D1][D2][D3][D4]  ← 4 data shards
                   [P1][P2]           ← 2 parity shards (XOR + RS math)
                    │   │   │   │
                drive0 drive1 drive2 drive3  (Docker volumes)
```

The math guarantees: **any 4 of the 6 shards are sufficient to reconstruct the object.**
That means losing 2 drives still leaves 4 surviving shards → full reconstruction.

| Drives lost | Shards remaining | Data recoverable? |
|-------------|-----------------|------------------|
| 0 | 6 / 6 | Yes (normal read) |
| 1 | 5 / 6 | Yes (EC reconstructs) |
| 2 | 4 / 6 | Yes (EC:2 tolerance limit) |
| 3 | 3 / 6 | No — insufficient shards |

Storage efficiency: **4 data shards / 6 total = 67%** (vs 33% for 3× replication).

In this single-node lab, the 4 drives are Docker volumes on one machine.
In a production cluster, shards are distributed across **different physical nodes**,
so drive AND node failures are both tolerated.


---
## 4 · Resilience in Action

Read all objects again to confirm they remain accessible.
In a real scenario with a failed drive, RustFS would transparently reconstruct
each object from surviving shards — the code below is identical to a healthy read.


In [ ]:
print('Reading all objects (transparent EC reconstruction if needed):\n')
for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    response = s3.get_object(Bucket=BUCKET, Key=key)
    body = response['Body'].read().decode('utf-8')
    print(f'  {key}: {body}')

print('\n✅ All objects accessible — fault tolerance verified.')


---
## 5 · Cleanup


In [ ]:
# Delete all objects
response = s3.list_objects_v2(Bucket=BUCKET)
for obj in response.get('Contents', []):
    s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
    print(f'🗑️  Deleted {obj["Key"]}')

# Delete the bucket
s3.delete_bucket(Bucket=BUCKET)
print(f'🗑️  Deleted bucket: {BUCKET}')

print('\n✅ Cleanup complete!')


---
## 📋 Summary

| Concept | Value in this lab |
|---------|------------------|
| EC scheme | Reed-Solomon (4+2) |
| Data shards | 4 |
| Parity shards | 2 |
| Max drive failures tolerated | 2 |
| Storage efficiency | 4/6 = 67% |
| vs 3× replication | 67% vs 33% (2× more efficient) |

### To simulate a real drive failure

You could pause a volume from the host and observe that reads still succeed:
```bash
# Pause drive1 (DO NOT do this in production!)
docker exec rustfs-server chmod 000 /data/drive1
# ... reads still work ...
docker exec rustfs-server chmod 755 /data/drive1
```

### Next steps

- **Lab 05**: Object versioning — protect data from accidental overwrites
- **Lab 06**: Erasure Coding deep dive — efficiency vs fault tolerance trade-offs
